# Chapter 9 §9.4: Breaking News Researcher with Recursive Summarization

This notebook accompanies Chapter 9 §9.4 of *Context Engineering with DSPy*: **Breaking News Researcher with Recursive Summarization**.

**Required environment variables (set in your `.env`):**

- `OPENAI_API_KEY`
- `SERPER_API_KEY`

**Service dependency:** This notebook is designed to be paired with [Serper](https://serper.dev/) (Google Search API) for live web search. The shipped version uses a simulated `web_search` function with pre-built results so the notebook runs without `SERPER_API_KEY`. To wire in real search, replace `web_search` with a Serper call (see the function docstring for the swap).

## Overview

```
9.4 Breaking News Researcher with Recursive Summarization

Demonstrates three context management patterns working together:
1. Web search retrieval — gather raw information from multiple sources
2. Chunking as Context Editing — break large results into digestible pieces
3. Recursive summarization — summarize chunks, then summarize summaries

Three failure modes compound here:
- Context Overflow: 50k+ tokens of search results exceed useful context
- Lost in the Middle: model ignores content in the middle of long context
- Context Fragmentation: related facts scattered across 10 different sources

The fix: search -> chunk -> summarize -> synthesize. Each stage reduces
context size while preserving signal.

Failure mode: Context Overflow + Lost in the Middle + Context Fragmentation
Technique: RAG + Context Summarization + Context Editing
Agentic pattern: Prompt Chaining (search -> chunk -> summarize -> synthesize)

Usage:
    python news_researcher.py                              # full demo
    python news_researcher.py --topic "AI chip export ban"  # custom topic
    python news_researcher.py --skip-optimization          # skip optimization
```


In [ ]:
%pip install -r ../requirements.txt -q

## Imports and LM configuration

In [ ]:
import os
from dotenv import load_dotenv

import dspy
from dspy.evaluate import Evaluate

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
dspy.configure(lm=lm)

## Environment check (Serper)

The live `web_search` swap requires `SERPER_API_KEY`. The shipped simulated version doesn't — this cell prints which path you're on.

In [ ]:
if not os.getenv("SERPER_API_KEY"):
    print("SERPER_API_KEY not set — using simulated web_search results.")
else:
    print("SERPER_API_KEY found — you can swap web_search() for a real Serper call.")

## Simulated web search (replace with real API in production)

In [ ]:
def web_search(query: str, num_results: int = 5) -> str:
    """Search the web for information on a topic.

    In production, replace with: Google Custom Search, Bing Search API,
    Tavily, or Serper. Returns raw search results as text.

    Args:
        query: Search query string
        num_results: Number of results to return

    Returns:
        Formatted search results with snippets
    """
    # Simulated search results — each topic has pre-built results
    # to make the demo reproducible without API keys
    results_db = {
        "quantum computing": [
            "Source: MIT Technology Review\nGoogle's quantum processor Willow achieved a computation in under 5 minutes that would take a classical supercomputer 10 septillion years. The 105-qubit chip demonstrates exponential error reduction as more qubits are added, a key milestone for quantum error correction.",
            "Source: Nature\nIBM's 1,121-qubit Condor processor launched in December 2023, but qubit count alone doesn't determine usefulness. IBM is now pivoting to modular quantum systems that link smaller processors. Their Heron chip focuses on lower error rates rather than more qubits.",
            "Source: Financial Times\nQuantum computing startups raised $1.2B in 2023, down from $2.1B in 2022. Investors are shifting from pure hardware plays to quantum software and applications. PsiQuantum's $450M raise was the largest single round.",
            "Source: ArXiv preprint\nResearchers at Harvard and QuEra demonstrated 48 logical qubits with error correction on a neutral atom quantum computer, achieving two-qubit gate fidelities above 99.5%. This approach may scale more readily than superconducting qubits.",
            "Source: Reuters\nChina's quantum computing program achieved a new benchmark with Jiuzhang 3.0, a photonic quantum computer that can solve specific problems 10^24 times faster than classical computers. The system uses 255 photons.",
        ],
        "AI regulation": [
            "Source: European Commission\nThe EU AI Act entered into force on August 1, 2024. High-risk AI systems must comply with transparency, data governance, and human oversight requirements. Penalties can reach 7% of global revenue.",
            "Source: White House\nPresident Biden's Executive Order on AI Safety (October 2023) requires developers of powerful AI systems to share safety test results with the government. NIST is developing AI risk management standards.",
            "Source: Wall Street Journal\nCalifornia's SB 1047 bill requiring safety testing for large AI models was vetoed by Governor Newsom in September 2024, citing concerns it would stifle innovation and drive companies out of state.",
            "Source: Bloomberg\nChina's interim measures for generative AI require providers to submit security assessments before launching services. Over 100 AI models have been approved under the system since August 2023.",
            "Source: Tech Policy Press\nThe UK's AI Safety Institute published its first evaluation of frontier models, testing GPT-4, Claude, and Gemini for dangerous capabilities. Results showed all models could be manipulated to produce harmful content with sufficient effort.",
        ],
        "climate technology": [
            "Source: BloombergNEF\nGlobal investment in energy transition technologies reached $1.77 trillion in 2023, with solar accounting for $380B. Battery storage investment doubled year-over-year to $36B.",
            "Source: Nature Energy\nPerovskite-silicon tandem solar cells achieved 33.7% efficiency in lab conditions, surpassing the theoretical limit of silicon-only cells (29.4%). Commercial production expected by 2026.",
            "Source: International Energy Agency\nGlobal CO2 emissions from energy reached 37.4 billion tonnes in 2023, a new record. However, emissions growth slowed to 1.1% as renewable deployment accelerated.",
            "Source: Carbon Brief\nDirect air capture costs dropped from $600/tonne to $250/tonne over 3 years, but remain far above the $100/tonne threshold needed for economic viability at scale.",
            "Source: Reuters\nThe US Inflation Reduction Act has driven $110B in clean energy manufacturing announcements since passage. Battery and EV manufacturing account for 60% of the total.",
        ],
    }

    # Find the closest matching topic
    query_lower = query.lower()
    for topic_key, results in results_db.items():
        if any(word in query_lower for word in topic_key.split()):
            return "\n\n".join(results[:num_results])

    # Default fallback
    return (
        "Source: General News\n"
        f"Search results for '{query}' would appear here. "
        "In production, this calls a real search API."
    )

## Signatures

In [ ]:
class GenerateSearchQueries(dspy.Signature):
    """Break a research topic into 3-5 focused search queries.

    Good queries are specific, non-overlapping, and cover different angles:
    technical details, market impact, key players, and recent developments.
    """

    topic: str = dspy.InputField(desc="Research topic or breaking news headline")
    queries: list[str] = dspy.OutputField(desc="3-5 focused search queries covering different angles")

class SummarizeChunk(dspy.Signature):
    """Extract the key facts from a chunk of search results.

    Focus on: specific numbers, dates, names, and claims.
    Drop: filler text, repeated information, and vague statements.
    Preserve: source attribution for every fact.
    """

    chunk: str = dspy.InputField(desc="A chunk of search results to summarize")
    topic: str = dspy.InputField(desc="The research topic (for relevance filtering)")
    summary: str = dspy.OutputField(desc="Key facts with source attribution, 2-3 sentences")

class SynthesizeReport(dspy.Signature):
    """Synthesize chunk summaries into a coherent research brief.

    Structure the output as:
    1. Headline (one line)
    2. Executive summary (2-3 sentences)
    3. Key findings (bulleted list with sources)
    4. Open questions (what we still don't know)

    Do NOT add information not present in the summaries.
    """

    topic: str = dspy.InputField(desc="Original research topic")
    summaries: str = dspy.InputField(desc="Summaries from individual chunks")
    report: str = dspy.OutputField(desc="Structured research brief")

## Chunking utilities

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks.

    Overlap prevents information loss at chunk boundaries.
    This is Context Editing (Ch 1.5.1) — restructuring content
    so the model processes manageable pieces.
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap

    return chunks

## News Researcher module

In [ ]:
class NewsResearcher(dspy.Module):
    """Multi-stage research pipeline: search -> chunk -> summarize -> synthesize.

    This module chains four stages to manage context effectively:
    1. Generate targeted search queries (avoid vague searches)
    2. Execute searches and chunk results (Context Editing)
    3. Summarize each chunk independently (Context Summarization)
    4. Synthesize summaries into a final report
    """

    def __init__(self):
        super().__init__()
        self.gen_queries = dspy.Predict(GenerateSearchQueries)
        self.summarize = dspy.ChainOfThought(SummarizeChunk)
        self.synthesize = dspy.ChainOfThought(SynthesizeReport)

    def forward(self, topic):
        # Stage 1: Generate search queries
        queries = self.gen_queries(topic=topic).queries

        # Stage 2: Search and chunk
        all_chunks = []
        for query in queries:
            results = web_search(query)
            chunks = chunk_text(results, chunk_size=200, overlap=30)
            all_chunks.extend(chunks)

        # Stage 3: Summarize each chunk
        summaries = []
        for chunk in all_chunks:
            result = self.summarize(chunk=chunk, topic=topic)
            summaries.append(result.summary)

        # Stage 4: Synthesize into final report
        combined = "\n\n".join(f"[{i+1}] {s}" for i, s in enumerate(summaries))
        report = self.synthesize(topic=topic, summaries=combined)

        return dspy.Prediction(
            queries=queries,
            num_chunks=len(all_chunks),
            num_summaries=len(summaries),
            report=report.report,
        )

## Evaluation metrics

In [ ]:
# Ground truth reports for evaluation
TOPICS = [
    {
        "topic": "Recent breakthroughs in quantum computing",
        "expected_entities": ["Google", "IBM", "Willow", "qubit", "error correction"],
        "expected_coverage": ["hardware advances", "error rates", "investment"],
    },
    {
        "topic": "Global AI regulation landscape in 2024",
        "expected_entities": ["EU AI Act", "Biden", "California", "China"],
        "expected_coverage": ["regulation", "safety testing", "penalties"],
    },
    {
        "topic": "Climate technology investment trends",
        "expected_entities": ["solar", "battery", "IRA", "perovskite"],
        "expected_coverage": ["investment amounts", "efficiency", "emissions"],
    },
]

def build_dataset():
    """Create evaluation dataset from topic definitions."""
    examples = [
        dspy.Example(
            topic=t["topic"],
            expected_entities=t["expected_entities"],
            expected_coverage=t["expected_coverage"],
        ).with_inputs("topic")
        for t in TOPICS
    ]
    return examples[:2], examples[2:]

def coverage_metric(example, pred, trace=None):
    """Measure how many expected entities and topics appear in the report.

    This is a proxy for information completeness — did the chunking
    and summarization pipeline preserve the key facts?
    """
    report_lower = pred.report.lower()

    # Entity coverage: are key names/terms mentioned?
    entities = getattr(example, "expected_entities", [])
    entity_hits = sum(1 for e in entities if e.lower() in report_lower)
    entity_score = entity_hits / len(entities) if entities else 1.0

    # Topic coverage: are main themes addressed?
    topics = getattr(example, "expected_coverage", [])
    topic_hits = sum(1 for t in topics if t.lower() in report_lower)
    topic_score = topic_hits / len(topics) if topics else 1.0

    # Penalize very short reports (likely incomplete)
    length_penalty = min(1.0, len(pred.report) / 200)

    return (0.4 * entity_score + 0.4 * topic_score + 0.2 * length_penalty)

## Demos

In [ ]:
def demo_naive_vs_chunked():
    """Show why chunking matters: naive dump vs. chunked summarization."""
    print("=" * 60)
    print("NAIVE vs. CHUNKED APPROACH")
    print("=" * 60)

    topic = "Recent breakthroughs in quantum computing"

    # Naive: dump all search results into one prompt
    naive_summarize = dspy.ChainOfThought(
        "search_results, topic -> report: str"
    )

    all_results = web_search("quantum computing", num_results=5)
    naive_result = naive_summarize(search_results=all_results, topic=topic)

    print(f"\nNaive approach (all results in one prompt):")
    print(f"  Input tokens: ~{len(all_results.split())} words")
    print(f"  Report: {naive_result.report[:200]}...")

    # Chunked: break into pieces, summarize each, then synthesize
    researcher = NewsResearcher()
    chunked_result = researcher(topic=topic)

    print(f"\nChunked approach (search -> chunk -> summarize -> synthesize):")
    print(f"  Queries generated: {len(chunked_result.queries)}")
    print(f"  Chunks processed: {chunked_result.num_chunks}")
    print(f"  Summaries created: {chunked_result.num_summaries}")
    print(f"  Report: {chunked_result.report[:200]}...")

def demo_research_pipeline():
    """Full research pipeline demo across all topics."""
    print("\n" + "=" * 60)
    print("RESEARCH PIPELINE DEMO")
    print("=" * 60)

    researcher = NewsResearcher()

    for topic_data in TOPICS:
        topic = topic_data["topic"]
        result = researcher(topic=topic)

        # Evaluate coverage
        example = dspy.Example(
            topic=topic,
            expected_entities=topic_data["expected_entities"],
            expected_coverage=topic_data["expected_coverage"],
        )
        score = coverage_metric(example, result)

        print(f"\n--- {topic} ---")
        print(f"Queries: {result.queries}")
        print(f"Chunks: {result.num_chunks} | Summaries: {result.num_summaries}")
        print(f"Coverage score: {score:.3f}")
        print(f"Report:\n{result.report[:300]}...")

def demo_optimization():
    """Optimize the research pipeline for coverage."""
    print("\n" + "=" * 60)
    print("OPTIMIZATION FOR COVERAGE")
    print("=" * 60)

    trainset, testset = build_dataset()

    # Baseline
    baseline = NewsResearcher()
    evaluator = Evaluate(devset=trainset + testset, metric=coverage_metric, display_progress=False)
    baseline_score = float(evaluator(baseline))

    # Optimize
    from dspy.teleprompt import BootstrapFewShot

    optimizer = BootstrapFewShot(
        metric=coverage_metric,
        max_bootstrapped_demos=2,
        max_labeled_demos=1,
    )
    optimized = optimizer.compile(NewsResearcher(), trainset=trainset)
    optimized_score = float(evaluator(optimized))

    print(f"\n  Baseline coverage:  {baseline_score:.1f}%")
    print(f"  Optimized coverage: {optimized_score:.1f}%")
    print(f"  Delta:              {optimized_score - baseline_score:+.1f}%")
    print()
    print("  Better search queries + better summarization instructions")
    print("  = more key facts preserved through the pipeline.")

## Run the demo

The code below mirrors the `main()` block in the source script with the argparse CLI branches removed — it runs the full demo path end-to-end.

In [ ]:
# Full demo
demo_naive_vs_chunked()
demo_research_pipeline()

if not False:
    demo_optimization()